# Bars-and-stripes pattern detector on the SpikeEngine STDP FPGA

Adapted from this project's own `03_bars_and_stripes_tutorial.ipynb` (`board_variants/npu_stdp_dev/notebooks/tutorials/`), which builds this network purely in software with `SNN.simulate()`. This notebook builds the SAME 3-layer network and instead runs it on a real `spikeengine` board (or a software-only preview with `RUN_HARDWARE = False`), cross-checking every test pattern against the software reference.

Given 9 binary inputs (a 3x3 pixel grid), the network reports whether the enabled pixels form a single vertical line (**bar**) or a single horizontal line (**stripe**). It takes exactly 3 enabled pixels, aligned in one row or one column, to trigger a positive output.

Three layers:
- **Input** (9 neurons, threshold 0): one per pixel, fires immediately when that pixel is on.
- **Hidden** (6 neurons, threshold 2): one per row/column (3 stripe-detectors + 3 bar-detectors). Each hidden neuron synapses from its row's (or column's) 3 input neurons at weight 1, so it only exceeds threshold 2 when all 3 of its pixels are on.
- **Output** (2 neurons, threshold 0): stripe and bar, each driven by its 3 corresponding hidden neurons.

Plus a **cancel neuron** (threshold 3): fed by ALL 9 inputs at weight 1, so a legitimate 3-pixel bar/stripe pattern gives it membrane 3, which does NOT exceed threshold 3 (strict `>`) -- it only fires with 4 or more enabled pixels. It synapses onto both outputs at weight -3, strong enough to veto a spurious output firing from an over-full grid.

In [1]:
import copy

import numpy as np

from superneuromat import SNN
from superneuromat import spikeengine as se

## Build the network

In [2]:
# SuperNeuroMAT's default leak is float('inf') (full reset each tick) -- the board's
# fixed-point config registers can't represent infinity, so use a large-but-finite
# stand-in instead (see logic_gates.ipynb for the same fix).
LEAK = 1000.0


def build_bars_and_stripes(n=3):
    """n x n grid version of the tutorial's 3x3 network (default n=3 matches it exactly)."""
    net = SNN()
    inputs = [net.create_neuron(threshold=0, leak=LEAK).idx for _ in range(n * n)]
    hidden = [net.create_neuron(threshold=n - 1, leak=LEAK).idx for _ in range(2 * n)]
    outputs = [net.create_neuron(threshold=0, leak=LEAK).idx for _ in range(2)]   # [0]=stripe, [1]=bar
    cancel = net.create_neuron(threshold=n, leak=LEAK).idx

    # stripe detection: hidden[i] <- row i (inputs i*n .. i*n+n-1)
    for i in inputs:
        net.create_synapse(i, hidden[i // n], weight=1)
    # bar detection: hidden[n + j] <- column j (inputs j, j+n, j+2n, ...)
    for i in inputs:
        net.create_synapse(i, hidden[n + (i % n)], weight=1)
    # hidden -> output: stripe detectors [0:n] -> outputs[0], bar detectors [n:2n] -> outputs[1]
    for h in hidden:
        net.create_synapse(h, outputs[hidden.index(h) // n], weight=1)
    # cancel: every input feeds it, and it vetoes both outputs
    for i in inputs:
        net.create_synapse(i, cancel, weight=1)
    for o in outputs:
        net.create_synapse(cancel, o, weight=-n)
    return net, inputs, outputs, hidden, cancel

## Configuration

All-integer weights/thresholds again, so `frac_bits=0`. The network is 3 synapse-hops deep (input -> hidden -> output, and input -> cancel -> output in parallel), so a run needs 3 ticks: inputs fire at t=0, hidden+cancel neurons fire at t=1, outputs resolve at t=2.

In [3]:
RUN_HARDWARE = True      # False -> software reference only
BOARD = 'basys3'      # 'basys3' | 'sp701' | 'zcu104'
PORT  = 'auto'           # 'auto' autodetects, or e.g. 'COM17'
FRAC_BITS = 0             # plain integers -- no fixed-point scaling needed for this example
N = 3                     # 3x3 grid, matching the tutorial
STEPS = 3

In [4]:
dev = se.connect(port=PORT, board=BOARD) if RUN_HARDWARE else None
if dev is not None:
    dev.soft_reset()
    print('connected to', BOARD)

connected to basys3


## Test patterns

Every one of the 6 valid bar/stripe patterns (3 rows + 3 columns), plus a few negative cases: an empty grid, a diagonal (3 pixels, but not aligned), and an over-full grid (a full row PLUS one extra pixel, which should trip the cancel neuron and veto the output).

In [5]:
def pattern(cells_on, n=3):
    """cells_on: iterable of (row, col) pairs -> length n*n binary vector, row-major."""
    grid = [0] * (n * n)
    for r, c in cells_on:
        grid[r * n + c] = 1
    return grid


TEST_CASES = {
    'stripe row 0':  (pattern([(0, 0), (0, 1), (0, 2)]), (True, False)),
    'stripe row 1':  (pattern([(1, 0), (1, 1), (1, 2)]), (True, False)),
    'stripe row 2':  (pattern([(2, 0), (2, 1), (2, 2)]), (True, False)),
    'bar col 0':     (pattern([(0, 0), (1, 0), (2, 0)]), (False, True)),
    'bar col 1':     (pattern([(0, 1), (1, 1), (2, 1)]), (False, True)),
    'bar col 2':     (pattern([(0, 2), (1, 2), (2, 2)]), (False, True)),
    'empty grid':    (pattern([]), (False, False)),
    'diagonal':      (pattern([(0, 0), (1, 1), (2, 2)]), (False, False)),
    'over-full row': (pattern([(0, 0), (0, 1), (0, 2), (1, 0)]), (False, False)),
}

## Run + verify every test case

`expect` is `(stripe, bar)`. Same pattern as the logic-gates notebook: `load_network()` once, then a `soft_reset()` + reload before each case to keep every run independent.

In [6]:
def software_run(net, inputs, outputs, grid, steps):
    ref = copy.deepcopy(net)
    for i, v in zip(inputs, grid):
        if v:
            ref.add_spike(0, i, 1)
    ref.simulate(steps)
    st = np.array(ref.spike_train)[-1]
    return tuple(bool(st[o]) for o in outputs)


def hardware_run(dev, net, inputs, outputs, grid, frac_bits, steps):
    dev.soft_reset()
    se.load_network(dev, net, frac_bits=frac_bits)
    n_neurons = len(net.neuron_thresholds)
    sched = {0: {i: v for i, v in zip(inputs, grid) if v}}
    spikes = se.run_schedule(dev, sched, total_ticks=steps, frac_bits=frac_bits,
                             n_neurons=n_neurons)
    last = spikes[-1]
    return tuple(bool(last[o]) for o in outputs)


net, inputs, outputs, hidden, cancel = build_bars_and_stripes(N)
all_ok = True
for name, (grid, expect) in TEST_CASES.items():
    sw = software_run(net, inputs, outputs, grid, STEPS)
    sw_ok = sw == expect
    line = f'{name:16s} grid={grid} software={sw} expect={expect} ({"OK" if sw_ok else "WRONG"})'
    if RUN_HARDWARE:
        hw = hardware_run(dev, net, inputs, outputs, grid, FRAC_BITS, STEPS)
        hw_ok = hw == expect
        line += f' hardware={hw} ({"OK" if hw_ok else "WRONG"})'
        all_ok = all_ok and hw_ok
    all_ok = all_ok and sw_ok
    print(line)
print('ALL CASES PASS' if all_ok else 'MISMATCH FOUND')
assert all_ok

stripe row 0     grid=[1, 1, 1, 0, 0, 0, 0, 0, 0] software=(True, False) expect=(True, False) (OK) hardware=(True, False) (OK)


stripe row 1     grid=[0, 0, 0, 1, 1, 1, 0, 0, 0] software=(True, False) expect=(True, False) (OK) hardware=(True, False) (OK)


stripe row 2     grid=[0, 0, 0, 0, 0, 0, 1, 1, 1] software=(True, False) expect=(True, False) (OK) hardware=(True, False) (OK)


bar col 0        grid=[1, 0, 0, 1, 0, 0, 1, 0, 0] software=(False, True) expect=(False, True) (OK) hardware=(False, True) (OK)


bar col 1        grid=[0, 1, 0, 0, 1, 0, 0, 1, 0] software=(False, True) expect=(False, True) (OK) hardware=(False, True) (OK)


bar col 2        grid=[0, 0, 1, 0, 0, 1, 0, 0, 1] software=(False, True) expect=(False, True) (OK) hardware=(False, True) (OK)


empty grid       grid=[0, 0, 0, 0, 0, 0, 0, 0, 0] software=(False, False) expect=(False, False) (OK) hardware=(False, False) (OK)


diagonal         grid=[1, 0, 0, 0, 1, 0, 0, 0, 1] software=(False, False) expect=(False, False) (OK) hardware=(False, False) (OK)


over-full row    grid=[1, 1, 1, 1, 0, 0, 0, 0, 0] software=(False, False) expect=(False, False) (OK) hardware=(False, False) (OK)
ALL CASES PASS


## Cleanup

In [7]:
if dev is not None:
    dev.close()
    print('closed')

closed
